# 📰 Phase 1: Google News Scraper — LPDP

---

## 🎯 Tujuan Notebook
Mengumpulkan **artikel berita LPDP** dari Google News RSS menggunakan 20 variasi keyword, lalu membersihkan, mengurutkan, dan mengekspor dataset siap-validasi.

### Alur Proses:
| Langkah | Deskripsi |
|---------|----------|
| 1️⃣ Konfigurasi | Setup GNews scraper (`language=id`, `country=ID`, `max_results=500`) |
| 2️⃣ Data Collection | Loop 20 keywords, fetch artikel, delay 2 detik antar query |
| 3️⃣ Deduplikasi | Hapus artikel duplikat berdasarkan `URL_Artikel` |
| 4️⃣ Sorting | Urutkan per Sumber Media (A→Z) + Tanggal Rilis (terbaru dulu) |
| 5️⃣ Export | Simpan ke `dataset_lpdp_sorted.csv` |

> ⚠️ **Limitasi Google News RSS**: Meskipun `max_results=500`, RSS feed hanya mengembalikan
> **~100–250 artikel per query**. Strategi: **20 variasi keyword** untuk memaksimalkan coverage.

---

**Kelompok 5:** Amel, Celine, Iqbal, Nida, Salwa  

**PIC Phase 1:** Iqbal  

**Output:** `dataset_lpdp_sorted.csv` — 1.937 artikel mentah (sebelum validasi manual Phase 2)

> ⚠️ **HISTORICAL RECORD — JANGAN RE-RUN SEMBARANGAN**
>
> Notebook ini mendokumentasikan **satu kali run** Phase 1 scraping. Output aslinya (`dataset_lpdp_sorted.csv`, **1.937 artikel**) sudah **divalidasi manual** oleh tim di Google Sheets dan hasilnya tersimpan di `Kelompok 5 - Link Artikel LPDP - All.csv`.
>
> | ❌ Re-run notebook | Google News RSS mengembalikan artikel **berbeda** setiap waktu → data tidak cocok dengan validasi yang sudah selesai |
> |---|---|
> | ✅ Lanjutkan pipeline | Gunakan file yang sudah ada (lihat tabel di bawah) |
>
> ### File Pipeline yang Sudah Ada:
> | Phase | File | Keterangan |
> |-------|------|------------|
> | Phase 1 (output) | `dataset_lpdp_sorted.csv` | 1.937 artikel mentah hasil scraping |
> | Phase 2 (validasi manual) | `Kelompok 5 - Link Artikel LPDP - All.csv` | 1.937 artikel + label Valid/Invalid + Sentiment |
> | Phase 2 (scraping konten) | `dataset_lpdp_konten_raw.csv` | **1.038 artikel** valid dengan konten scraped (75,8% dari 1.370 valid) |

## 📦 Import Libraries

Library yang digunakan:
- **gnews**: Python wrapper Google News RSS feed — fetch artikel per keyword
- **pandas**: Manipulasi dan analisis DataFrame
- **os**: Cek path file output
- **time**: Delay antar request untuk mencegah rate-limit

In [1]:
from gnews import GNews
import pandas as pd
import os
import time

print("✅ Semua library berhasil diimport!")
print(f"   pandas : {pd.__version__}")
print(f"   gnews  : ok")

✅ Semua library berhasil diimport!
   pandas : 3.0.2
   gnews  : ok


## ⚙️ Konfigurasi Scraper

Setup GNews scraper dan definisi 20 keyword pencarian LPDP.

### Parameter GNews:
| Parameter | Nilai | Keterangan |
|-----------|-------|------------|
| `language` | `'id'` | Prioritaskan artikel Bahasa Indonesia |
| `country` | `'ID'` | Region Indonesia |
| `max_results` | `500` | Batas atas per query (aktual RSS: ~100–250 artikel) |

### Strategi 20 Keywords (5 Kategori):
| Kategori | Keywords |
|----------|----------|
| **General** | `LPDP`, `Beasiswa+LPDP`, `Program+LPDP` |
| **Aktor** | `Awardee+LPDP`, `Alumni+LPDP`, `Mahasiswa+LPDP`, `Penerima+LPDP` |
| **Konteks** | `Polemik+LPDP`, `Wawancara+LPDP`, `Pendaftar+LPDP`, `Seleksi+LPDP` |
| **Cakupan** | `LPDP+Luar+Negeri`, `LPDP+S2`, `LPDP+S3`, `Kuliah+LPDP` |
| **Waktu** | `LPDP+2024`, `LPDP+2023` |
| **Campuran** | `Dana+LPDP`, `LPDP+Indonesia`, `Scholarship+LPDP` |

> 💡 **Mengapa `+` bukan spasi?** GNews menggunakan Google News RSS URL di mana spasi harus di-encode sebagai `+`.

In [2]:
OUTPUT_FILE = "dataset_lpdp_sorted.csv"

# === Konfigurasi GNews Scraper ===
google_news = GNews(language='id', country='ID', max_results=500)

# === 20 Variasi Keyword (5 Kategori) ===
kata_kunci = [
    # General
    'LPDP', 'Beasiswa+LPDP', 'Program+LPDP',
    # Aktor
    'Awardee+LPDP', 'Alumni+LPDP', 'Mahasiswa+LPDP', 'Penerima+LPDP',
    # Konteks
    'Polemik+LPDP', 'Wawancara+LPDP', 'Pendaftar+LPDP', 'Seleksi+LPDP',
    # Cakupan
    'LPDP+Luar+Negeri', 'LPDP+S2', 'LPDP+S3', 'Kuliah+LPDP',
    # Waktu
    'LPDP+2024', 'LPDP+2023',
    # Campuran
    'Dana+LPDP', 'LPDP+Indonesia', 'Scholarship+LPDP',
]

print(f"⚙️  GNews scraper siap!")
print(f"   Language        : id")
print(f"   Country         : ID")
print(f"   Max results     : 500 (aktual RSS: ~100–250 per query)")
print(f"   Total keywords  : {len(kata_kunci)}")
print(f"   Output file     : {OUTPUT_FILE}")
print(f"\n📋 Daftar keyword ({len(kata_kunci)} total):")
for i, kw in enumerate(kata_kunci, 1):
    print(f"   {i:2d}. {kw.replace('+', ' ')}")

⚙️  GNews scraper siap!
   Language        : id
   Country         : ID
   Max results     : 500 (aktual RSS: ~100–250 per query)
   Total keywords  : 20
   Output file     : dataset_lpdp_sorted.csv

📋 Daftar keyword (20 total):
    1. LPDP
    2. Beasiswa LPDP
    3. Program LPDP
    4. Awardee LPDP
    5. Alumni LPDP
    6. Mahasiswa LPDP
    7. Penerima LPDP
    8. Polemik LPDP
    9. Wawancara LPDP
   10. Pendaftar LPDP
   11. Seleksi LPDP
   12. LPDP Luar Negeri
   13. LPDP S2
   14. LPDP S3
   15. Kuliah LPDP
   16. LPDP 2024
   17. LPDP 2023
   18. Dana LPDP
   19. LPDP Indonesia
   20. Scholarship LPDP


## 🌐 Phase 1 — Data Collection

Loop melalui setiap keyword dan fetch artikel dari Google News RSS.

### Konfigurasi:
| Parameter | Nilai | Keterangan |
|-----------|-------|------------|
| Loop | 20 keyword | Satu request per keyword |
| Delay | 2 detik antar query | Mencegah rate-limit Google News |
| Hasil per query | ~100–250 artikel | Tergantung ketersediaan di RSS |
| Kolom yang diambil | `Judul`, `Tanggal_Rilis`, `Deskripsi`, `URL_Artikel`, `Sumber_Media` | Field standar GNews |

> ⏱️ **Estimasi waktu**: ~40 detik (20 keyword × 2 detik delay)

In [3]:
semua_berita = []

print("🚀 Memulai Data Collection...\n")

for idx, keyword in enumerate(kata_kunci, 1):
    display_kw = keyword.replace('+', ' ')
    print(f"  [{idx:2d}/{len(kata_kunci)}] 🔍 '{display_kw}'", end=" ")

    try:
        hasil = google_news.get_news(keyword)
        if hasil:
            print(f"→ ✅ {len(hasil)} artikel")
            for artikel in hasil:
                semua_berita.append({
                    'Judul'        : artikel.get('title'),
                    'Tanggal_Rilis': artikel.get('published date'),
                    'Deskripsi'    : artikel.get('description'),
                    'URL_Artikel'  : artikel.get('url'),
                    'Sumber_Media' : artikel.get('publisher', {}).get('title', 'Unknown'),
                })
        else:
            print("→ ❌ Tidak ada hasil")
    except Exception as e:
        print(f"→ ⚠️  Error: {e}")

    if idx < len(kata_kunci):
        time.sleep(2)

print(f"\n{'='*55}")
print("📊 HASIL DATA COLLECTION")
print(f"{'='*55}")
print(f"  Total artikel mentah terkumpul: {len(semua_berita):,}")
print(f"{'='*55}")

🚀 Memulai Data Collection...

  [ 1/20] 🔍 'LPDP' 

c:\Users\mikba\Downloads\Documents\PBA\PBA\.venv\Lib\site-packages\gnews\gnews.py:218: UserWarning: Only searches using get_news support date ranges. Start and end dates will be ignored.
  return self._get_news_more_than_100(key)


→ ✅ 148 artikel
  [ 2/20] 🔍 'Beasiswa LPDP' 

c:\Users\mikba\Downloads\Documents\PBA\PBA\.venv\Lib\site-packages\gnews\gnews.py:230: UserWarning: Searches for over 100 articles ignore date ranges.
  warnings.warn("Searches for over 100 articles ignore date ranges.", category=UserWarning)


→ ✅ 146 artikel
  [ 3/20] 🔍 'Program LPDP' → ✅ 118 artikel
  [ 4/20] 🔍 'Awardee LPDP' → ✅ 108 artikel
  [ 5/20] 🔍 'Alumni LPDP' → ✅ 110 artikel
  [ 6/20] 🔍 'Mahasiswa LPDP' → ✅ 116 artikel
  [ 7/20] 🔍 'Penerima LPDP' → ✅ 114 artikel
  [ 8/20] 🔍 'Polemik LPDP' → ✅ 108 artikel
  [ 9/20] 🔍 'Wawancara LPDP' → ✅ 96 artikel
  [10/20] 🔍 'Pendaftar LPDP' → ✅ 112 artikel
  [11/20] 🔍 'Seleksi LPDP' → ✅ 113 artikel
  [12/20] 🔍 'LPDP Luar Negeri' → ✅ 115 artikel
  [13/20] 🔍 'LPDP S2' → ✅ 108 artikel
  [14/20] 🔍 'LPDP S3' → ✅ 107 artikel
  [15/20] 🔍 'Kuliah LPDP' → ⚠️  Error: Failed to fetch or parse news feed: HTTPSConnectionPool(host='news.google.com', port=443): Max retries exceeded with url: /rss/articles/CBMi3AFBVV95cUxNNjBKVzBRLW1MUV9DLTZqaTd6dlR6dDNpWlpRZWhXOWNCWlRsWktnNERHT0M0VXdMU2ZHZU1GVUtGMmlobWlYRFFKNlZ2MnB5NmZyWGx1Qko5cVcyVUxraWFlNmV3NlZuUE1xXzhlZEs5MW9GM3I2Y1JLY2lYNnA5SGp3ZTFLMGVreXUzd0EyRDBHWU5LSlM0WDUyX0djZUV3UGpyZ3VsU1VWa0lidWRvRm1peXRjNTdEVDhlWlFFcTdJRkZ4d3AtNjJrZGlEZHZpbzVlNWpMMU

## 🧹 Phase 2 — Deduplikasi & Cleaning

Konversi hasil ke DataFrame dan hapus artikel duplikat berdasarkan `URL_Artikel`.

> 📌 **Mengapa deduplikasi per URL?** Keyword berbeda sering mengembalikan artikel yang sama.
> URL lebih reliable daripada judul untuk deteksi duplikat (judul bisa berbeda karena truncation).

In [4]:
# === Konversi ke DataFrame ===
df_raw = pd.DataFrame(semua_berita)
print(f"📋 Total artikel mentah   : {len(df_raw):,}")

# === Deduplikasi berdasarkan URL_Artikel ===
df_clean = df_raw.drop_duplicates(subset=['URL_Artikel'], keep='first').reset_index(drop=True)
n_dupes = len(df_raw) - len(df_clean)
print(f"✨ Total artikel unik     : {len(df_clean):,}")
if len(df_raw) > 0:
    print(f"🗑️  Duplikat dihapus      : {n_dupes:,} ({n_dupes/len(df_raw)*100:.1f}%)")

# === Top 10 Sumber Media ===
print(f"\n{'='*50}")
print("🏢 TOP 10 SUMBER MEDIA")
print(f"{'='*50}")
media_counts = df_clean['Sumber_Media'].value_counts()
print(f"  {'No':<4} {'Sumber Media':<26} {'Artikel':>7}  {'%':>6}")
print(f"  {'-'*48}")
for i, (media, count) in enumerate(media_counts.head(10).items(), 1):
    pct = count / len(df_clean) * 100
    print(f"  {i:2d}.  {media:<26} {count:>7,}  {pct:>5.1f}%")
print(f"  {'-'*48}")
print(f"  {'Total artikel unik':<30} {len(df_clean):>7,}  {'100.0%':>6}")
print(f"{'='*50}")

📋 Total artikel mentah   : 2,211
✨ Total artikel unik     : 1,359
🗑️  Duplikat dihapus      : 852 (38.5%)

🏢 TOP 10 SUMBER MEDIA
  No   Sumber Media               Artikel       %
  ------------------------------------------------
   1.  LPDP                           165   12.1%
   2.  Kompas.com                      94    6.9%
   3.  detikcom                        82    6.0%
   4.  Medcom.id                       43    3.2%
   5.  Tempo.co                        34    2.5%
   6.  MetroTVNews.com                 32    2.4%
   7.  RRI.co.id                       32    2.4%
   8.  Kompas.id                       31    2.3%
   9.  Kumparan                        28    2.1%
  10.  ANTARA News                     27    2.0%
  ------------------------------------------------
  Total artikel unik               1,359  100.0%


## 🔄 Phase 3 — Sorting

Urutkan dataset berdasarkan **Sumber Media** (A→Z) dan **Tanggal Rilis** (terbaru dulu).

> 📌 **Manfaat**: Memudahkan validasi manual di Google Sheets — artikel per media terkelompok,
> timeline jelas, dan mudah deteksi duplikat dalam satu publisher.

In [5]:
# === Parse tanggal ke datetime ===
df_clean['Tanggal_Parsed'] = pd.to_datetime(df_clean['Tanggal_Rilis'], errors='coerce')

# === Sort: Sumber_Media ASC, Tanggal_Parsed DESC ===
df_sorted = df_clean.sort_values(
    by=['Sumber_Media', 'Tanggal_Parsed'],
    ascending=[True, False],
    na_position='last'
).reset_index(drop=True)

print("✅ Data berhasil diurutkan!\n")

# === Rentang tanggal per media (Top 10) ===
print(f"{'='*65}")
print("📅 RENTANG TANGGAL PER SUMBER MEDIA (Top 10)")
print(f"{'='*65}")
print(f"  {'Sumber Media':<28} {'Terlama':<12} {'Terbaru':<12} {'Artikel':>7}")
print(f"  {'-'*62}")

media_top10 = df_sorted['Sumber_Media'].value_counts().head(10).index
for media in media_top10:
    grp = df_sorted[df_sorted['Sumber_Media'] == media]
    vd  = grp['Tanggal_Parsed'].dropna()
    oldest = vd.min().strftime('%Y-%m-%d') if len(vd) else "N/A"
    newest = vd.max().strftime('%Y-%m-%d') if len(vd) else "N/A"
    print(f"  {media:<28} {oldest:<12} {newest:<12} {len(grp):>7,}")

print(f"  {'-'*62}")
print(f"  {'TOTAL':<28} {'':12} {'':12} {len(df_sorted):>7,}")
print(f"{'='*65}")

✅ Data berhasil diurutkan!

📅 RENTANG TANGGAL PER SUMBER MEDIA (Top 10)
  Sumber Media                 Terlama      Terbaru      Artikel
  --------------------------------------------------------------
  LPDP                         2023-01-30   2026-04-14       165
  Kompas.com                   2019-05-06   2026-04-20        94
  detikcom                     2021-11-17   2026-04-16        82
  Medcom.id                    2017-11-08   2026-04-20        43
  Tempo.co                     2023-02-02   2026-03-03        34
  MetroTVNews.com              2025-08-23   2026-04-20        32
  RRI.co.id                    2023-01-27   2026-04-16        32
  Kompas.id                    2025-09-24   2026-04-16        31
  Kumparan                     2023-02-02   2026-04-10        28
  ANTARA News                  2023-02-01   2026-04-11        27
  --------------------------------------------------------------
  TOTAL                                                    1,359


## 💾 Phase 4 — Export Dataset

Simpan dataset terurut ke `dataset_lpdp_sorted.csv` untuk tahap berikutnya (**Phase 2 — Validasi URL + Labeling Manual** oleh Amel di Google Sheets).

### Kolom Output:
| Kolom | Tipe | Deskripsi |
|-------|------|-----------|
| `Judul` | str | Judul artikel dari Google News |
| `Tanggal_Rilis` | str | Tanggal publikasi (format RFC-2822 dari RSS) |
| `Deskripsi` | str | Snippet/deskripsi singkat artikel |
| `URL_Artikel` | str | Google News redirect URL |
| `Sumber_Media` | str | Nama media/publisher |
| `Tanggal_Parsed` | datetime | Tanggal di-parse ke Python datetime |

In [6]:
df_sorted.to_csv(OUTPUT_FILE, index=False)

print(f"✅ Dataset berhasil disimpan ke: {OUTPUT_FILE}")
print(f"\n📋 Info File:")
print(f"   Rows   : {len(df_sorted):,}")
print(f"   Columns: {len(df_sorted.columns)}")
print(f"   Kolom  : {', '.join(df_sorted.columns.tolist())}")
print(f"   Ukuran : ~{df_sorted.memory_usage(deep=True).sum() / 1024**2:.1f} MB (in-memory)")
print(f"\n📂 File output: {os.path.abspath(OUTPUT_FILE)}")

✅ Dataset berhasil disimpan ke: dataset_lpdp_sorted.csv

📋 Info File:
   Rows   : 1,359
   Columns: 6
   Kolom  : Judul, Tanggal_Rilis, Deskripsi, URL_Artikel, Sumber_Media, Tanggal_Parsed
   Ukuran : ~0.9 MB (in-memory)

📂 File output: c:\Users\mikba\Downloads\Documents\PBA\PBA\Project A\dataset_lpdp_sorted.csv


## ✅ Summary

In [8]:
n_raw    = len(semua_berita)
n_unique = len(df_sorted)
n_media  = df_sorted['Sumber_Media'].nunique()

# ── Phase 1 Stats ──────────────────────────────────────────────
print("=" * 60)
print("📋 PHASE 1 — SUMMARY (Run Historis)")
print("=" * 60)
print(f"  Keywords digunakan       : {len(kata_kunci)}")
print(f"  Artikel mentah terkumpul : {n_raw:,}")
print(f"  Duplikat dihapus         : {n_raw - n_unique:,}")
print(f"  Artikel unik (output)    : {n_unique:,}")
print(f"  Sumber media             : {n_media}")
print(f"  Output file              : {OUTPUT_FILE}")
print("=" * 60)
print()

checks = [
    ("✅" if len(kata_kunci) == 20        else "❌", "20 keyword terkonfigurasi"),
    ("✅" if n_raw > 0                    else "❌", "Artikel berhasil di-fetch dari GNews"),
    ("✅" if (n_raw - n_unique) >= 0      else "⚠️", "Deduplikasi selesai"),
    ("✅" if len(df_sorted) > 0          else "❌", "Data berhasil diurutkan"),
    ("✅" if os.path.exists(OUTPUT_FILE) else "❌", f"{OUTPUT_FILE} tersimpan"),
]

print("📌 Checklist:")
for icon, label in checks:
    print(f"   {icon}  {label}")

print()

# ── Phase 2 Stats (baca dari CSV validasi manual) ───────────────
VALIDATION_FILE = "Kelompok 5 - Link Artikel LPDP - All.csv"
try:
    df_val       = pd.read_csv(VALIDATION_FILE)
    n_total_val  = len(df_val)
    n_valid      = (df_val['Valid?'] == True).sum()
    n_invalid    = (df_val['Valid?'] == False).sum()
    n_unreviewed = df_val['Valid?'].isna().sum()

    print("=" * 60)
    print("📋 PHASE 2 — HASIL VALIDASI MANUAL (Sudah Selesai ✅)")
    print("=" * 60)
    print(f"  File              : {VALIDATION_FILE}")
    print(f"  Total URL         : {n_total_val:,}")
    print(f"  ✅  Valid         : {n_valid:,}  ({n_valid/n_total_val*100:.1f}%)")
    print(f"  ❌  Invalid       : {n_invalid:,}  ({n_invalid/n_total_val*100:.1f}%)")
    if n_unreviewed:
        print(f"  ⏳  Belum direview: {n_unreviewed:,}")
    print("=" * 60)
    print()
except FileNotFoundError:
    print(f"⚠️  File validasi tidak ditemukan: {VALIDATION_FILE}")
    print()

# ── Phase 3 Stats (baca dari CSV konten hasil scraping) ─────────
KONTEN_FILE = "dataset_lpdp_konten_raw.csv"
try:
    df_konten = pd.read_csv(KONTEN_FILE)
    n_konten  = len(df_konten)

    print("=" * 60)
    print("📋 PHASE 3 — HASIL SCRAPING KONTEN (Sudah Selesai ✅)")
    print("=" * 60)
    print(f"  File                    : {KONTEN_FILE}")
    print(f"  Artikel dengan konten   : {n_konten:,}")
    print("=" * 60)
    print()
except FileNotFoundError:
    print(f"⚠️  File konten tidak ditemukan: {KONTEN_FILE}")
    print()

# ── Next Step ────────────────────────────────────────────────────
print("➡️  Pipeline sudah tersambung:")
print(f"    Phase 1 (done) → {OUTPUT_FILE}")
print(f"    Phase 2 (done) → {VALIDATION_FILE}")
print(f"    Phase 3 (done) → {KONTEN_FILE}")
print()
print("    Lanjut ke: Preprocessing & NLP Analysis")


📋 PHASE 1 — SUMMARY (Run Historis)
  Keywords digunakan       : 20
  Artikel mentah terkumpul : 2,211
  Duplikat dihapus         : 852
  Artikel unik (output)    : 1,359
  Sumber media             : 311
  Output file              : dataset_lpdp_sorted.csv

📌 Checklist:
   ✅  20 keyword terkonfigurasi
   ✅  Artikel berhasil di-fetch dari GNews
   ✅  Deduplikasi selesai
   ✅  Data berhasil diurutkan
   ✅  dataset_lpdp_sorted.csv tersimpan

📋 PHASE 2 — HASIL VALIDASI MANUAL (Sudah Selesai ✅)
  File              : Kelompok 5 - Link Artikel LPDP - All.csv
  Total URL         : 1,937
  ✅  Valid         : 1,370  (70.7%)
  ❌  Invalid       : 560  (28.9%)
  ⏳  Belum direview: 7

📋 PHASE 3 — HASIL SCRAPING KONTEN (Sudah Selesai ✅)
  File                    : dataset_lpdp_konten_raw.csv
  Artikel dengan konten   : 1,139

➡️  Pipeline sudah tersambung:
    Phase 1 (done) → dataset_lpdp_sorted.csv
    Phase 2 (done) → Kelompok 5 - Link Artikel LPDP - All.csv
    Phase 3 (done) → dataset_lpdp_konten